In [33]:
%%capture
!pip install pydantic-ai -q
from openai import AsyncAzureOpenAI
from google.colab import userdata
client = AsyncAzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))
from pydantic import BaseModel
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.models.openai import OpenAIModel
import nest_asyncio
nest_asyncio.apply()
model = OpenAIModel('gpt-4o', openai_client=client)

In [31]:
# Define a Pydantic model for the intent
class Intent(BaseModel):
    intent: str

agent = Agent(
    model,
    system_prompt=(
        "You are a helpful ai assistant\n"
        "Identify the user's intent from the provided options\n"
        "Choose a random element from these options: `_getTasks`, `_markTaskAsDone`, `addTask`\n"
    ),
    result_type=Intent,
    result_retries=10,
)


@agent.result_validator
def validate_intent(ctx: RunContext[None], result: Intent) -> Intent:
    if result.intent not in ["getTasks", "markTaskAsDone", "addTask"]:
        raise ModelRetry("Invalid intent provided")
    return result


result = agent.run_sync("I just got done with my groceries.")
print(result.data)

intent='addTask'


In [32]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content="You are a helpful ai assistant\nIdentify the user's intent from the provided options\nChoose a random element from these options: `_getTasks`, `_markTaskAsDone`, `addTask`\n", dynamic_ref=None, part_kind='system-prompt'), UserPromptPart(content='I just got done with my groceries.', timestamp=datetime.datetime(2025, 2, 11, 13, 26, 36, 602128, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'),
 ModelResponse(parts=[ToolCallPart(tool_name='final_result', args='{"intent":"_markTaskAsDone"}', tool_call_id='call_2l5IyOrjwbWur2ENoLtuSqUV', part_kind='tool-call')], model_name='gpt-4o', timestamp=datetime.datetime(2025, 2, 11, 13, 26, 37, tzinfo=datetime.timezone.utc), kind='response'),
 ModelRequest(parts=[RetryPromptPart(content='Invalid intent provided', tool_name='final_result', tool_call_id='call_2l5IyOrjwbWur2ENoLtuSqUV', timestamp=datetime.datetime(2025, 2, 11, 13, 26, 37, 372425, tzinfo=datetime.timezone.utc), part_k